# GSE249479 | Distinguish molecular responses to TNF and LPS

**Results-first notebook, from actual public-data ScGeo 0.2.1 outputs.** No H5AD download, fitting or ScGeo installation is required to explore these results. The [full molecular workflow](../../v0.2.0/06_gse249479_shift_explanations.ipynb) and [geometry workflow](../../v0.2.0/04_gse249479_inflammation.ipynb) remain separately reproducible.

Source: [GEO GSE249479](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE249479). This analysis uses the frozen representation of 34,432 retained cells, with seven detailed marker-inferred state definitions (including Ambiguous HSPC). PBS is the common reference. The original linear basis is uncentered SVD50, stored as X_pca50.

**No biological sample identities were available.** All 14 contrasts have status `insufficient_samples`. These are pooled-cell descriptive estimates, not population-level inference. No pseudo-replicates, bootstrap intervals, p-values, or sample-level stability claims are supplied. The selected genes and figures are retrospective illustrations, not independent confirmation.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
folder = next((p for p in (cwd, cwd / "notebooks/applications/results/gse249479")
               if (p / "selected_results.json").is_file()), None)
if folder is None:
    raise RuntimeError("Open from the notebook directory or repository root.")
raw = (folder / "selected_results.json").read_bytes()
assert hashlib.sha256(raw).hexdigest() == "11457cf462777afc82cf123a89c8d4408b964a254980637b4d8d66646853dcf4"
data = json.loads(raw)
def label(frame):
    frame["state"] = frame["state_idx"].map(dict(enumerate(data["states"])))
    if "contrast_idx" in frame:
        frame["contrast"] = frame["contrast_idx"].map(dict(enumerate(data["contrasts"])))
    return frame
summary = label(pd.DataFrame(data["summary"], columns=data["summary_columns"]))
direction = pd.DataFrame(data["directions"], columns=data["direction_columns"])
direction.insert(0, "state", data["states"])
genes = label(pd.DataFrame(data["genes"], columns=data["gene_columns"]))
genes["gene"] = genes["gene_idx"].map(dict(enumerate(data["gene_names"])))
programs = label(pd.DataFrame(data["programs"], columns=["state_idx","contrast_idx",*data["program_names"]]))
v, audit = data["validation"], data["mapping_audit"]
assert (len(summary), len(direction), len(genes), len(programs)) == (14,7,36,14)
assert not summary.duplicated(["state_idx","contrast_idx"]).any()
assert v["biological_samples"] == v["bootstrap_draws"] == 0
assert v["status"] == "insufficient_samples"
assert v["max_frozen_error"] == 0
assert audit["max_abs_residual"] <= audit["tolerance"]
assert (summary.delta_norm - summary.contribution_sum).abs().max() < 1e-6
assert ((genes.expression_delta * genes.axis_loading - genes.contribution).abs() < 3e-9).all()
display(pd.Series({"state-contrast rows": len(summary),
                   "full gene estimates audited": v["gene_rows"],
                   "full program estimates audited": v["program_rows"],
                   "global affine residual (reported)": audit["max_abs_residual"],
                   "global acceptance threshold": audit["tolerance"],
                   "max gene-sum error (unrounded)": v["max_sum_error"],
                   "biological replicate identities": v["biological_samples"],
                   "conditional bootstrap draws": v["bootstrap_draws"]}))

## 1. Magnitudes and directions

![TNF and LPS magnitudes and directional cosines](figures/01_geometry.svg)

[Open full-size SVG](figures/01_geometry.svg)

![Selected common-origin vector schematics](figures/02_direction.svg)

[Open full-size SVG](figures/02_direction.svg)

Activated HSCs have similar observed displacement magnitudes under TNF and LPS, yet a directional cosine near 0.52. The lymphoid-labeled state has nearly orthogonal directions. These are point estimates, not angular significance tests.

Direction cosines were calculated from all 3,000 exported gene-axis loadings aligned by original feature ID. Because the original SVD loading columns are nearly orthonormal, normalized dot products approximate the original coordinate-space angles. Only the derived cosines and norms are retained in this compact snapshot. The full archive or analysis rerun is required to recompute them.

In [ ]:
direction["angle_degrees"] = direction.cosine.map(lambda v: math.degrees(math.acos(v)))
display(summary.pivot(index="state", columns="contrast", values="delta_norm").reindex(data["states"]).round(3))
display(direction.round(3))
display(summary[["state","contrast","n_cells0","n_cells1"]])
# The PBS reference is reused in both contrasts; do not add both PBS counts as independent cells.

## 2. Signed gene contributions are not expression changes

![Selected gene expression changes and signed geometric contributions](figures/03_genes.svg)

[Open full-size SVG](figures/03_genes.svg)

AC105402.3 contributes positively in both contrasts for the three focus states despite decreasing expression. HLA-DQA1 and STAT1 have larger observed TNF contributions than LPS in quiescent HSCs. XIST contributions motivate checking sample composition and technical covariates; these data do not permit adjustment for biological replicates or donor effects.

These are weighted expression-center contrasts, not conventional log fold changes. For a gene, the contribution is its expression-center contrast times the loading along the fitted unit direction. Positive values support that observed displacement and negative values oppose it. Summing over the full HVG space reconstructs the net displacement. Six selected genes are only a subset of the 3,000 features.

In [ ]:
display(genes[["state","contrast","gene","expression_delta","axis_loading","contribution"]].round(6))

## 3. A signed contribution can change sign even when expression change does not

![FOS expression and signed contributions under TNF and LPS](figures/04_fos.svg)

[Open full-size SVG](figures/04_fos.svg)

FOS expression decreases for all six focus-state contrasts. Its TNF contributions are small and negative, whereas its LPS contributions are positive. This follows from the contrast-specific loading directions. The sign reversal is a numerical illustration of the API, not a statistically established switch or opposite causal gene function.

In [ ]:
display(genes[genes.gene.eq("FOS")][["state","contrast","expression_delta","axis_loading","contribution"]].round(7))

## 4. Study-derived and predefined molecular programs

![Selected HVG-restricted program contributions](figures/05_programs.svg)

[Open full-size SVG](figures/05_programs.svg)

The original study HSC-IM top-50 subset (39 measured genes from the frozen HSC-II ranked marker definition) contributes more to the activated-HSC TNF shift than its LPS shift; quiescent HSCs and MPPs show the opposite descriptive pattern. These are overlapping sums of gene contributions, not enrichment scores or evidence of persistent inflammatory memory.

All 18 usable program sums are retained in the table below. The apoptosis program was excluded because only one requested gene was retained among the HVGs. Absence from the basis does not establish absence of gene expression or pathway activity.

In [ ]:
coverage = pd.DataFrame({"program":data["program_names"],"measured_hvgs":data["program_sizes"],
                         "requested_members":data["program_requested"]})
display(coverage)
display(programs.set_index(["state","contrast"])[data["program_names"]].T.round(4))
focus = programs[programs.state_idx.lt(3)].merge(summary[["state_idx","contrast_idx","delta_norm"]],
                                                on=["state_idx","contrast_idx"],validate="one_to_one")
focus["selected_program_signed_sum_over_net_pct"] = 100 * focus.study_hsc_im_top50_subset / focus.delta_norm
display(focus[["state","contrast","study_hsc_im_top50_subset","delta_norm",
               "selected_program_signed_sum_over_net_pct"]].round(3))

## 5. What this example reproduces

The source archive was audited against the complete state/contrast roster, frozen displacement, gene-product and program-sum conservation, feature identities and absence of biological sample intervals. This compact notebook reproduces summary calculations and reads deterministic static plots, not the full expression-to-SVD mapping or the original cell-level curves. No raw counts or pooled pseudo-sample rows are included.

Run `node scripts/render_public_hspc_gallery.mjs --check` from the repository root to verify the five committed SVGs. See the [gallery guide](README.md) for exact source provenance and the archive-extraction command. The [replicate-aware lung results notebook](../gse211713/GSE211713_fibroblast_results.ipynb) complements this example with independent mice and conditional sample-bootstrap summaries.